In [ ]:
import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Video, display
import matplotlib
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_everything()

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

def show_video(filename, width=700):
    # OpenCV writes mp4v; re-encode to H.264, which browsers and PowerPoint play
    tmp = f'{filename[:-4]}_H264{filename[-4:]}'
    os.system(f'ffmpeg -loglevel error -y -i {filename} -vcodec libx264 -pix_fmt yuv420p {tmp}')
    os.remove(filename)
    os.rename(tmp, filename)
    display(Video(filename, embed=True, width=width))

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image.jpg https://github.com/chrirupp/cv_course/raw/main/data/image.jpg
wget --no-verbose --output-document=dog_cutout.png https://github.com/chrirupp/cv_course/raw/main/data/dog_cutout.png
wget --no-verbose --output-document=running.mp4 https://github.com/chrirupp/cv_course/raw/main/data/running.mp4
wget --no-verbose --output-document=cat_door.mp4 https://github.com/chrirupp/cv_course/raw/main/data/cat_door.mp4
wget --no-verbose --output-document=poodles.mp4 https://github.com/chrirupp/cv_course/raw/main/data/poodles.mp4
wget --no-verbose --output-document=vlm_cat_answers.json https://github.com/chrirupp/cv_course/raw/main/data/vlm_cat_answers.json

# Lecture 12 - Video

What makes video different from images, and how to test whether a model actually uses time: redundancy between frames,
a per-frame classifier, two video models on shuffled and reversed clips, motion and optical flow, what the first layer
of a 3D CNN learns, and why video transformers use tubelets and factorised attention.

We use two short **real** clips from Wikimedia Commons and a **synthetic** clip that we render ourselves, so that we know
the true motion of every pixel.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large
pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

## Two real clips

* `running.mp4`: the first shot of ["Running.gif"](https://commons.wikimedia.org/wiki/File:Running.gif) by Fengalon
  (public domain), 20 frames at 10 FPS. A Kinetics-400 class: *jogging*.
* `cat_door.mp4`: seconds 18.4 to 22.2 of ["Cat opening door"](https://commons.wikimedia.org/wiki/File:Cat_opening_door.webm)
  by Astropulse (CC BY-SA 4.0), 114 frames at 30 FPS. An action with a direction: played backwards, the cat *closes* the door.

At the end we also use `poodles.mp4` from Lecture 11 (["Two poodles playing"](https://commons.wikimedia.org/wiki/File:Two_poodles_playing.ogv)
by Living in Monrovia (A W), CC BY-SA 2.0).

In [ ]:
def read_video(filename):
    cap = cv2.VideoCapture(filename)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    return np.stack(frames), fps

running, running_fps = read_video('running.mp4')
cat, cat_fps = read_video('cat_door.mp4')
print('running:', running.shape, running_fps, 'FPS')
print('cat:    ', cat.shape, cat_fps, 'FPS')

fig, axs = plt.subplots(2, 6, figsize=(18, 7), gridspec_kw=dict(height_ratios=[1, 1.4]))
for ax, t in zip(axs[0], np.linspace(0, len(running) - 1, 6).astype(int)):
    ax.imshow(running[t]); ax.set_title(f'running, frame {t}')
for ax, t in zip(axs[1], np.linspace(0, len(cat) - 1, 6).astype(int)):
    ax.imshow(cat[t]); ax.set_title(f'cat, frame {t}')
for ax in axs.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

## Redundancy

At 30 FPS, consecutive frames are nearly identical: only the pixels on the moving cat (and the door) change. Even a
second later most of the image is the same. Video models exploit this: they subsample in time, merge neighbouring frames
into one token (tubelets) and, for self-supervised learning, can mask out 90% of a clip and still reconstruct it.

In [ ]:
t, later = 50, 80                          # frame t, t + 1 and t + 30 (one second later)
grey = cat.astype(np.float32).mean(axis=3)
diff1 = np.abs(grey[t + 1] - grey[t])
diff30 = np.abs(grey[later] - grey[t])
consecutive = np.abs(np.diff(grey, axis=0))
print(f'consecutive frames: mean |difference| {consecutive.mean():.1f} grey levels, '
      f'{(consecutive > 10).mean():.1%} of the pixels change by more than 10')
print(f'frames {t} and {t + 1}: {(diff1 > 10).mean():.1%} of the pixels change by more than 10')
print(f'frames {t} and {later}: {(diff30 > 10).mean():.1%} of the pixels change by more than 10')

fig, axs = plt.subplots(1, 4, figsize=(16, 7.5), gridspec_kw=dict(wspace=0.08))
axs[0].imshow(cat[t]); axs[0].set_title(f'frame {t}', fontsize=18)
axs[1].imshow(cat[t + 1]); axs[1].set_title(f'frame {t + 1}', fontsize=18)
axs[2].imshow(diff1, cmap='magma', vmin=0, vmax=60); axs[2].set_title(f'|frame {t + 1} − frame {t}|', fontsize=18)
axs[3].imshow(diff30, cmap='magma', vmin=0, vmax=60); axs[3].set_title(f'|frame {later} − frame {t}|', fontsize=18)
for ax in axs:
    ax.axis('off')
savefig('video_redundancy')
plt.show()

## A per-frame classifier

The simplest video model ignores time: classify every frame with an image model and average the probabilities. Here the
image model is CLIP (ViT-B/16) used zero-shot: each frame is compared with the text "a photo of a person *label*." for
the 400 Kinetics classes. CLIP has never seen a video, and the average over the frames is the same whatever their order.

In [ ]:
from transformers import CLIPModel, CLIPProcessor, VideoMAEConfig
kinetics_labels = list(VideoMAEConfig.from_pretrained('MCG-NJU/videomae-base-finetuned-kinetics').id2label.values())
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch16').to(device).eval()
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch16')

def clip_per_frame(frames, labels):
    with torch.no_grad():
        text = clip_processor(text=[f'a photo of a person {l}.' for l in labels], return_tensors='pt', padding=True).to(device)
        images = clip_processor(images=list(frames), return_tensors='pt')['pixel_values'].to(device)
        return clip(**text, pixel_values=images).logits_per_image.softmax(dim=-1).cpu().numpy()   # (frames, labels)

idx16 = np.linspace(0, len(running) - 1, 16).round().astype(int)     # 16 frames, as for the video models below
probs = clip_per_frame(running[idx16], kinetics_labels)
for t, p in zip(idx16, probs):
    print(f'frame {t:2d}: {kinetics_labels[p.argmax()]} ({p.max():.2f})')
average = probs.mean(axis=0)
top = average.argsort()[::-1][:5]
print('average:', [(kinetics_labels[k], round(float(average[k]), 2)) for k in top])
n_jogging = (probs.argmax(axis=1) == kinetics_labels.index('jogging')).sum()
print(f'"jogging" is the top class in {n_jogging} of 16 frames')

show = [0, 5, 10, 15]                   # 4 of the 16 frames
fig, axs = plt.subplots(1, 5, figsize=(22, 4.6), gridspec_kw=dict(width_ratios=[1, 1, 1, 1, 1.5], wspace=0.12))
for ax, i in zip(axs, show):
    p = probs[i]
    ax.imshow(running[idx16[i]])
    ax.set_title(f'frame {idx16[i]}: {kinetics_labels[p.argmax()]} ({p.max():.2f})', fontsize=16)
    ax.axis('off')
ax = axs[4]
ax.barh(range(5)[::-1], average[top], color='#2a6fdb', height=0.6)
ax.set_yticks([])
for y, k in zip(range(5)[::-1], top):
    ax.text(average[k] + 0.015, y, f'{average[k]:.2f}  {kinetics_labels[k]}', va='center', fontsize=15)
ax.set_xlim(0, 1)
ax.set_title('average of 16 frames', fontsize=16)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(axis='x', labelsize=12)
savefig('video_per_frame')
plt.show()

## Does the model use time?

Two VideoMAE models (ViT-B, 16 frames, tubelets of 2x16x16) with the same architecture, fine-tuned on two datasets:

* **Kinetics-400**: 400 classes that mostly say *what* happens ("jogging", "playing guitar").
* **Something-Something v2**: 174 classes that say *how* objects are moved ("Opening [something]",
  "Closing [something]", "Pushing [something] from left to right").

We give each model the clip, the same 16 frames in a random order, and the clip played backwards. A model whose
prediction does not change when the order changes does not use the order. Shuffling also creates fast jumps that no
real video has; reversing keeps the motion natural and only flips its direction.

(With transformers 5 the checkpoints' `q_bias`/`v_bias` are not loaded any more; `load_videomae` copies them in.)

In [ ]:
from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor
from transformers.utils import logging
logging.set_verbosity_error()          # no load report for the attention biases (see the note above)
from huggingface_hub import hf_hub_download

def load_videomae(name):
    model = VideoMAEForVideoClassification.from_pretrained(name)
    if not hasattr(model.videomae.encoder.layer[0].attention.attention, 'q_bias'):
        try:
            from safetensors.torch import load_file
            state = load_file(hf_hub_download(name, 'model.safetensors'))
        except Exception:
            state = torch.load(hf_hub_download(name, 'pytorch_model.bin'), map_location='cpu')
        with torch.no_grad():
            for i, layer in enumerate(model.videomae.encoder.layer):
                attention = layer.attention.attention
                attention.query.bias.copy_(state[f'videomae.encoder.layer.{i}.attention.attention.q_bias'])
                attention.key.bias.zero_()
                attention.value.bias.copy_(state[f'videomae.encoder.layer.{i}.attention.attention.v_bias'])
    return model.to(device).eval(), VideoMAEImageProcessor.from_pretrained(name)

def predict(model_and_processor, frames, k=3):
    model, processor = model_and_processor
    x = processor(list(frames), return_tensors='pt')['pixel_values'].to(device)
    with torch.no_grad():
        p = model(x).logits.softmax(dim=-1)[0].cpu()
    values, indices = p.topk(k)
    return [(model.config.id2label[int(i)], float(v)) for v, i in zip(values, indices)]

kinetics = load_videomae('MCG-NJU/videomae-base-finetuned-kinetics')
ssv2 = load_videomae('MCG-NJU/videomae-base-finetuned-ssv2')

rng = np.random.default_rng(0)
shuffle = rng.permutation(16)
orders = {'forward': np.arange(16), 'shuffled': shuffle, 'reversed': np.arange(16)[::-1]}
cat16 = np.linspace(0, len(cat) - 1, 16).round().astype(int)

results = {}
for clip_name, frames, idx in [('running', running, idx16), ('cat', cat, cat16)]:
    for model_name, model in [('Kinetics-400', kinetics), ('SSv2', ssv2)]:
        for order_name, order in orders.items():
            top3 = predict(model, frames[idx[order]])
            results[clip_name, model_name, order_name] = top3
            print(f'{clip_name:8s} {model_name:13s} {order_name:9s}', ', '.join(f'{l} ({p:.2f})' for l, p in top3))
        print()

In [ ]:
# the Kinetics model on the running clip: forward, shuffled, reversed (4 of the 16 input frames, in the order the model sees them)
def tile(frames, gap=4):
    # 2x2 tile with dark grey gaps
    h, w = frames[0].shape[:2]
    out = np.full((2 * h + gap, 2 * w + gap, 3), 90, np.uint8)
    for k, f in enumerate(frames[:4]):
        r, c = divmod(k, 2)
        out[r * (h + gap):r * (h + gap) + h, c * (w + gap):c * (w + gap) + w] = f
    return out

fig, axs = plt.subplots(1, 3, figsize=(21, 6.6), gridspec_kw=dict(wspace=0.1))
for ax, (order_name, order) in zip(axs, orders.items()):
    picked = order[[0, 5, 10, 15]]
    ax.imshow(tile(running[idx16[picked]]))
    for k, i in enumerate(picked):
        r, c = divmod(k, 2)
        ax.text(c * 414 + 10, r * 312 + 40, f'{k * 5 + 1}: frame {idx16[i]}', color='white', fontsize=14,
                bbox=dict(facecolor='black', alpha=0.6, pad=2, edgecolor='none'))
    label, p = results['running', 'Kinetics-400', order_name][0]
    ax.set_title(f'{order_name}: {label} ({p:.2f})', fontsize=22)
    ax.axis('off')
savefig('video_kinetics_test')
plt.show()

In [ ]:
# the Something-Something model on the cat clip: a side-by-side video (forward, reversed, shuffled) for the slide
font_path = os.path.join(matplotlib.get_data_path(), 'fonts', 'ttf', 'DejaVuSans.ttf')
font, font_bold = ImageFont.truetype(font_path, 19), ImageFont.truetype(font_path.replace('Sans.ttf', 'Sans-Bold.ttf'), 22)

def ssv2_label(order_name):
    label, p = results['cat', 'SSv2', order_name][0]
    return f'{label.replace("[something]", "something")} ({p:.2f})'

crop = slice(96, None)                     # drop the wall above the door handle
VH, VW = cat[0][crop].shape[:2]
gap, header = 10, 64
panels = [('forward', np.arange(len(cat))), ('reversed', np.arange(len(cat))[::-1]),
          ('shuffled', cat16[np.repeat(shuffle, int(np.ceil(len(cat) / 16)))[:len(cat)]])]   # the 16 shuffled inputs, each held
if SAVE_FIGURES:
    os.makedirs('figures', exist_ok=True)
path = 'figures/video_reverse.mp4' if SAVE_FIGURES else 'video_reverse.mp4'
size = (3 * VW + 2 * gap, VH + header)
writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), 15, size)   # half speed
for t in range(len(cat)):
    canvas = Image.new('RGB', size, 'white')
    draw = ImageDraw.Draw(canvas)
    for k, (name, frame_index) in enumerate(panels):
        x = k * (VW + gap)
        canvas.paste(Image.fromarray(cat[frame_index[t]][crop]), (x, header))
        draw.text((x + 4, 2), name, fill='black', font=font_bold)
        draw.text((x + 4, 32), ssv2_label(name), fill=(30, 60, 160), font=font)
    frame = np.asarray(canvas)
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    if t == 0:
        poster = frame
        if SAVE_FIGURES:
            cv2.imwrite('figures/video_reverse_poster.png', cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()
show_video(path)

### The arrow of time

Can you tell which of these two rows plays forwards? Here the answer is in the physics of the scene (a door does not
swing open by itself while the cat walks backwards), not in any single frame. Predicting the direction of time is
itself a classic video task and a self-supervised training signal (Pickup et al., CVPR 2014; Wei et al., CVPR 2018).

In [ ]:
pick = np.linspace(0, len(cat) - 1, 6).round().astype(int)
fig, axs = plt.subplots(2, 6, figsize=(18, 7.4), gridspec_kw=dict(wspace=0.04, hspace=0.14))
for row, (label, order) in zip(axs, [('A', pick), ('B', pick[::-1])]):
    for ax, i in zip(row, order):
        ax.imshow(cat[i][96:]); ax.axis('off')
    row[0].text(-0.12, 0.5, label, transform=row[0].transAxes, fontsize=30, fontweight='bold', ha='center', va='center')
savefig('video_arrow_of_time')
plt.show()
print('A plays forwards, B backwards')

## A synthetic clip with exact ground truth

In [ ]:
# A synthetic video with exact ground truth: a camera pans and zooms over a photo (the background),
# a dog walks through the scene and passes behind a pole that stands in the scene.
# Because we render it ourselves, we know where every pixel goes (flow) and when it is hidden (occlusion).
background = cv2.cvtColor(cv2.imread('image.jpg'), cv2.COLOR_BGR2RGB).astype(np.float32)
cutout = cv2.imread('dog_cutout.png', cv2.IMREAD_UNCHANGED)
cutout = cv2.resize(cutout, (0, 0), fx=0.75, fy=0.75, interpolation=cv2.INTER_AREA)
dog_rgb = cv2.cvtColor(cutout[:, :, :3], cv2.COLOR_BGR2RGB).astype(np.float32)
dog_alpha = cutout[:, :, 3].astype(np.float32) / 255

H, W, T = 360, 480, 48
POLE_X, POLE_WIDTH = 640, 36            # the pole, in background (world) coordinates

def camera(t):
    '''scale and offset: background point X appears at s * (X - o) in frame t'''
    return 0.5 * (1 + 0.003 * t), np.array([40 + 5.0 * t, 60 + 1.0 * t])

def dog_position(t):
    '''top-left corner of the dog in frame t (image coordinates)'''
    return np.array([20 + 6.0 * t, 120 + 8 * np.sin(t / 5)])

def layers(t):
    '''the masks of the dog and the pole in frame t (the pole is in front of the dog)'''
    s, o = camera(t)
    px, py = dog_position(t)
    shift = np.float32([[1, 0, px], [0, 1, py]])
    dog_mask = cv2.warpAffine(dog_alpha, shift, (W, H))
    pole_mask = np.zeros((H, W), np.float32)
    x0, x1 = s * (POLE_X - o[0]), s * (POLE_X + POLE_WIDTH - o[0])
    pole_mask[:, max(int(round(x0)), 0):max(int(round(x1)), 0)] = 1
    return dog_mask, pole_mask, shift

def render(t):
    s, o = camera(t)
    A = np.float32([[s, 0, -s * o[0]], [0, s, -s * o[1]]])
    frame = cv2.warpAffine(background, A, (W, H), flags=cv2.INTER_LINEAR)
    dog_mask, pole_mask, shift = layers(t)
    dog = cv2.warpAffine(dog_rgb, shift, (W, H))
    frame = frame * (1 - dog_mask[..., None]) + dog * dog_mask[..., None]
    frame = frame * (1 - pole_mask[..., None]) + np.array([70, 60, 55]) * pole_mask[..., None]
    return np.clip(frame, 0, 255).astype(np.uint8)

def ground_truth_tracks(points, t0=0):
    '''positions (T, N, 2) and visibility (T, N) of points given in frame t0'''
    points = np.asarray(points, np.float32)
    dog_mask, pole_mask, _ = layers(t0)
    xi = np.clip(points[:, 0].round().astype(int), 0, W - 1)
    yi = np.clip(points[:, 1].round().astype(int), 0, H - 1)
    on_dog = dog_mask[yi, xi] > 0.5
    s0, o0 = camera(t0)
    world = points / s0 + o0                    # background points in world coordinates
    tracks, visible = np.zeros((T, len(points), 2), np.float32), np.zeros((T, len(points)), bool)
    for t in range(T):
        s, o = camera(t)
        tracks[t] = np.where(on_dog[:, None], points - dog_position(t0) + dog_position(t), s * (world - o))
        dog_mask, pole_mask, _ = layers(t)
        x, y = tracks[t, :, 0], tracks[t, :, 1]
        inside = (x >= 0) & (x <= W - 1) & (y >= 0) & (y <= H - 1)
        xi, yi = np.clip(x.round().astype(int), 0, W - 1), np.clip(y.round().astype(int), 0, H - 1)
        hidden = (pole_mask[yi, xi] > 0.5) | (~on_dog & (dog_mask[yi, xi] > 0.5))
        visible[t] = inside & ~hidden
    return tracks, visible

def ground_truth_flow(t):
    '''dense optical flow from frame t to frame t+1 (H, W, 2)'''
    ys, xs = np.mgrid[0:H, 0:W].astype(np.float32)
    points = np.stack([xs.ravel(), ys.ravel()], axis=1)
    s0, o0 = camera(t)
    s1, o1 = camera(t + 1)
    flow = s1 * (points / s0 + o0 - o1) - points
    dog_mask, pole_mask, _ = layers(t)
    flow[dog_mask.ravel() > 0.5] = dog_position(t + 1) - dog_position(t)
    flow[pole_mask.ravel() > 0.5] = s1 * (points[pole_mask.ravel() > 0.5] / s0 + o0 - o1) - points[pole_mask.ravel() > 0.5]
    return flow.reshape(H, W, 2).astype(np.float32)

video = np.stack([render(t) for t in range(T)])
print('video:', video.shape)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate(frames, interval=80):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    im = ax.imshow(frames[0])
    ax.axis('off')
    anim = FuncAnimation(fig, lambda t: [im.set_data(frames[t])], frames=len(frames), interval=interval)
    plt.close()
    return HTML(anim.to_jshtml())

animate(video)

## Motion

The simplest motion cue is the difference between consecutive frames. It responds to moving edges, but it does not tell us *where* things move. Note that the camera moves too, so the whole background changes.

In [ ]:
ts = [10, 26]
fig, axs = plt.subplots(len(ts), 3, figsize=(18, 4.5 * len(ts)))
for row, t in zip(axs, ts):
    diff = np.abs(video[t + 1].astype(np.float32) - video[t].astype(np.float32)).mean(axis=2)
    row[0].imshow(video[t])
    row[0].set_title(f'Frame {t}')
    row[1].imshow(video[t + 1])
    row[1].set_title(f'Frame {t + 1}')
    row[2].imshow(diff, cmap='magma')
    row[2].set_title('|difference|')
    for ax in row:
        ax.axis('off')
plt.show()

## Optical flow

Optical flow assigns a 2D displacement to every pixel. We compare a classical method (Farnebäck, polynomial expansion + coarse-to-fine) with RAFT, a learned method that iteratively refines the flow using a 4D correlation volume. Flow is visualised with the usual colour wheel: the hue gives the direction, the saturation the magnitude. How these methods work is the topic of Lecture 13.

The error measure is the *end-point error* (EPE): the mean Euclidean distance between estimated and true flow vectors.

In [ ]:
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
from torchvision.utils import flow_to_image
raft = raft_large(weights=Raft_Large_Weights.DEFAULT).to(device).eval()

def raft_flow(frame0, frame1):
    # RAFT expects images in [-1, 1] with sides divisible by 8
    x0 = torch.from_numpy(frame0).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(device)
    x1 = torch.from_numpy(frame1).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(device)
    with torch.no_grad():
        return raft(x0, x1)[-1][0].permute(1, 2, 0).cpu().numpy()   # the last of 12 refinement iterations

def farneback_flow(frame0, frame1):
    g0, g1 = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in (frame0, frame1)]
    return cv2.calcOpticalFlowFarneback(g0, g1, None, pyr_scale=0.5, levels=4, winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)

def flow_rgb(flow, max_flow=12):
    flow = torch.from_numpy(flow).permute(2, 0, 1).clamp(-max_flow, max_flow)
    flow[0, 0, 0], flow[1, 0, 1] = max_flow, max_flow       # fix the colour scale across images
    return flow_to_image(flow).permute(1, 2, 0).numpy()

def epe(flow, gt):
    return np.linalg.norm(flow - gt, axis=2).mean()

t = 26
gt = ground_truth_flow(t)
flows = {'Ground truth': gt, 'Farnebäck': farneback_flow(video[t], video[t + 1]), 'RAFT': raft_flow(video[t], video[t + 1])}
fig, axs = plt.subplots(1, 4, figsize=(24, 4.5))
axs[0].imshow(video[t])
axs[0].set_title(f'Frame {t}', fontsize=20)
for ax, (name, flow) in zip(axs[1:], flows.items()):
    ax.imshow(flow_rgb(flow))
    ax.set_title(name if name == 'Ground truth' else f'{name}: EPE {epe(flow, gt):.2f} px', fontsize=20)
for ax in axs:
    ax.axis('off')
savefig('video_optical_flow')
plt.show()

In [ ]:
# EPE over the whole clip
errors = {'Farnebäck': [], 'RAFT': []}
raft_flows, gt_flows = [], []
for t in tqdm(range(T - 1)):
    gt = ground_truth_flow(t)
    raft_flows.append(raft_flow(video[t], video[t + 1]))
    gt_flows.append(gt)
    errors['Farnebäck'].append(epe(farneback_flow(video[t], video[t + 1]), gt))
    errors['RAFT'].append(epe(raft_flows[-1], gt))
plt.figure(figsize=(10, 4))
for name, e in errors.items():
    plt.plot(e, label=f'{name} (mean {np.mean(e):.2f} px)')
plt.xlabel('frame')
plt.ylabel('end-point error (px)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# the clip and its flow side by side as a video: input | RAFT | ground truth (same colour wheel)
font_path = os.path.join(matplotlib.get_data_path(), 'fonts', 'ttf', 'DejaVuSans.ttf')
font_title = ImageFont.truetype(font_path, 24)

def write_panels(path, panel_frames, titles, fps, header=40, gap=12):
    '''panel_frames: list of arrays (n_frames, h, w, 3) with the same shape; writes an mp4v video and returns frame 0'''
    n, h, w = panel_frames[0].shape[:3]
    size = (len(panel_frames) * w + (len(panel_frames) - 1) * gap, h + header)
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, size)
    first = None
    for t in range(n):
        canvas = Image.new('RGB', size, 'white')
        draw = ImageDraw.Draw(canvas)
        for k, (frames_k, title) in enumerate(zip(panel_frames, titles)):
            canvas.paste(Image.fromarray(frames_k[t]), (k * (w + gap), header))
            draw.text((k * (w + gap) + 4, 6), title, fill='black', font=font_title)
        frame = np.asarray(canvas)
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
        first = frame if first is None else first
    writer.release()
    return first

if SAVE_FIGURES:
    os.makedirs('figures', exist_ok=True)
path = 'figures/video_flow.mp4' if SAVE_FIGURES else 'video_flow.mp4'
poster = write_panels(path, [video[:T - 1], np.stack([flow_rgb(f) for f in raft_flows]), np.stack([flow_rgb(f) for f in gt_flows])],
                      ['input', 'RAFT', 'ground truth'], fps=8)
if SAVE_FIGURES:
    cv2.imwrite('figures/video_flow_poster.png', cv2.cvtColor(poster, cv2.COLOR_RGB2BGR))
show_video(path, width=900)

In [ ]:
# the colour code of the flow images, drawn with the same flow_rgb: every pixel of the disc gets the flow vector from
# the centre to that pixel (up to 12 px, the saturation point used above); image y points down
R = 200
ys, xs = np.mgrid[-R:R + 1, -R:R + 1].astype(np.float32)
inside = xs ** 2 + ys ** 2 <= R ** 2
wheel_flow = np.stack([xs, ys], axis=2) * 12 / R * inside[..., None]
wheel = flow_rgb(wheel_flow)
wheel[~inside] = 255

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(wheel, extent=(-R, R, R, -R))
for (dx, dy), name in [((1, 0), 'right'), ((-1, 0), 'left'), ((0, -1), 'up'), ((0, 1), 'down')]:
    ax.annotate('', xy=(dx * R * 0.8, dy * R * 0.8), xytext=(0, 0), arrowprops=dict(arrowstyle='->', lw=2.5, color='black'))
    ax.text(dx * R * (1.3 if dx else 1.13), dy * R * 1.13, name, ha='center', va='center', fontsize=20)
ax.set_xlim(-R * 1.6, R * 1.6); ax.set_ylim(R * 1.3, -R * 1.3)
ax.axis('off')
savefig('video_flow_wheel')
plt.show()

## What does a 3D CNN learn?

The first layer of a 3D ResNet-18 trained on Kinetics-400 has 64 filters of size $3 \times 7 \times 7$ (time x height x width). Many filters are the same in all three time steps (appearance), others change over time: they respond to motion in a particular direction.

In [ ]:
from torchvision.models.video import r3d_18, R3D_18_Weights
r3d = r3d_18(weights=R3D_18_Weights.DEFAULT)
w = r3d.stem[0].weight.detach()          # (64, 3 colour channels, 3 time steps, 7, 7)
print('first layer:', tuple(w.shape))

# show every filter as three 7x7 frames side by side (one colour normalisation per filter)
# filters that look the same in all three frames respond to appearance; filters that change respond to motion or flicker
temporal_change = (w[:, :, 1:] - w[:, :, :-1]).flatten(1).norm(dim=1) / w.flatten(1).norm(dim=1)
order = temporal_change.argsort()
fig, axs = plt.subplots(8, 8, figsize=(16, 8))
for ax, idx in zip(axs.ravel(), order):
    f = w[idx].permute(1, 2, 3, 0).numpy()          # (time, 7, 7, colour)
    f = (f - f.min()) / (f.max() - f.min())
    strip = np.concatenate([f[0], np.ones((7, 1, 3)), f[1], np.ones((7, 1, 3)), f[2]], axis=1)
    ax.imshow(strip, interpolation='nearest')
    ax.axis('off')
fig.suptitle('First layer of a 3D ResNet-18 (Kinetics-400): 64 filters, each shown as t=0 | t=1 | t=2,\nsorted from static (top left) to changing over time (bottom right)')
plt.show()

In [ ]:
# the same filters as an animation: each filter is played through its three time steps, over and over
# (4 rows of 16, sorted from static, top left, to changing over time, bottom right; each 7x7 filter enlarged 9x)
scale, gap, rows, cols = 9, 6, 4, 16
cell = 7 * scale
size = (cols * cell + (cols - 1) * gap, rows * cell + (rows - 1) * gap)
grids = []
for t in range(3):
    grid = np.full((size[1], size[0], 3), 255, np.uint8)
    for k, idx in enumerate(order):
        f = w[idx].permute(1, 2, 3, 0).numpy()          # (time, 7, 7, colour), normalised over all time steps
        f = (f - f.min()) / (f.max() - f.min())
        tile_ = np.kron(f[t], np.ones((scale, scale, 1)))  # nearest-neighbour upsampling
        r, c = divmod(k, cols)
        grid[r * (cell + gap):r * (cell + gap) + cell, c * (cell + gap):c * (cell + gap) + cell] = (tile_ * 255).astype(np.uint8)
    grids.append(grid)

if SAVE_FIGURES:
    os.makedirs('figures', exist_ok=True)
path = 'figures/video_3d_filters.mp4' if SAVE_FIGURES else 'video_3d_filters.mp4'
writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), 10, size)
for repeat in range(20):                       # 0, 1, 2, 0, 1, 2, ...: each time step for 0.2 s
    for t in range(3):
        for _ in range(2):
            writer.write(cv2.cvtColor(grids[t], cv2.COLOR_RGB2BGR))
writer.release()
if SAVE_FIGURES:
    cv2.imwrite('figures/video_3d_filters_poster.png', cv2.cvtColor(grids[0], cv2.COLOR_RGB2BGR))
show_video(path, width=900)

## Tubelets and the cost of attention over space and time

A Video Vision Transformer (ViViT, VideoMAE) turns a clip into tokens by cutting it into *tubelets* (here $2 \times 16 \times 16$ pixels): one token per tubelet, so a 16-frame clip at $224^2$ gives $8 \times 14 \times 14 = 1568$ tokens. Joint space-time attention compares every token with every other token, which grows quadratically with the number of frames. Factorised attention first attends within each frame (spatial), then across frames at each location (temporal).

In [ ]:
# one tubelet: the same 16x16 patch in two consecutive frames becomes one token
def center_crop(frame, size=224):
    h, w = frame.shape[:2]
    s = size / min(h, w)
    frame = cv2.resize(frame, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    y, x = (frame.shape[0] - size) // 2, (frame.shape[1] - size) // 2
    return frame[y:y + size, x:x + size]

frames = [center_crop(running[idx16[k]]) for k in (0, 1)]
row, col = 5, 7                                 # the tubelet we highlight (on the runner)
tubelet = np.stack([f[row * 16:(row + 1) * 16, col * 16:(col + 1) * 16] for f in frames])
print('tubelet:', tubelet.shape, '=', tubelet.size, 'values -> one token;',
      f'16 frames at 224x224: {16 // 2} x {224 // 16} x {224 // 16} = {16 // 2 * (224 // 16) ** 2} tokens')

fig, axs = plt.subplots(1, 3, figsize=(18, 6.4), gridspec_kw=dict(width_ratios=[1, 1, 1], wspace=0.12))
for k, (ax, f) in enumerate(zip(axs, frames)):
    ax.imshow(f)
    for g in range(0, 225, 16):
        ax.axhline(g - 0.5, color='white', lw=0.6, alpha=0.7)
        ax.axvline(g - 0.5, color='white', lw=0.6, alpha=0.7)
    ax.add_patch(plt.Rectangle((col * 16 - 0.5, row * 16 - 0.5), 16, 16, fill=False, edgecolor='red', lw=3))
    ax.set_title(f'frame {idx16[k]}: 14×14 patches of 16×16', fontsize=17)
    ax.axis('off')
axs[2].imshow(np.concatenate([tubelet[0], np.full((16, 2, 3), 255, np.uint8), tubelet[1]], axis=1), interpolation='nearest')
axs[2].set_title('one tubelet: 2×16×16×3 = 1536 values\n→ one token (linear projection)', fontsize=17)
axs[2].axis('off')
savefig('video_tubelets')
plt.show()

In [ ]:
def attention_pairs(frames, size=224, patch=16, tubelet=2):
    n_t, n_s = frames // tubelet, (size // patch) ** 2
    joint = (n_t * n_s) ** 2
    factorised = n_t * n_s ** 2 + n_s * n_t ** 2
    return n_t * n_s, joint, factorised

frames = np.array([8, 16, 32, 64, 128, 256])
results_cost = np.array([attention_pairs(f) for f in frames])
for f, (tokens, joint, fact) in zip(frames, results_cost):
    print(f'{f:4d} frames: {tokens:6d} tokens, joint {joint/1e6:9.1f} M pairs, factorised {fact/1e6:7.1f} M pairs ({joint/fact:5.1f}x fewer)')

fig, ax = plt.subplots(figsize=(13, 4.6))
ax.loglog(frames, results_cost[:, 1], 'o-', color='#2a6fdb', lw=2, ms=8, label='joint space-time attention')
ax.loglog(frames, results_cost[:, 2], 'o-', color='#e07b28', lw=2, ms=8, label='factorised (spatial, then temporal)')
for f, (tokens, joint, fact) in zip(frames, results_cost):
    if f in (16, 256):
        ax.annotate(f'{joint / fact:.1f}× fewer', (f, np.sqrt(joint * fact)), ha='left', va='center', fontsize=15,
                    xytext=(10, 0), textcoords='offset points')
        ax.plot([f, f], [fact, joint], color='grey', lw=1, ls='--')
ax.set_xticks(frames)
ax.set_xticklabels([f'{f}\n({f // 2 * 196} tokens)' for f in frames], fontsize=13)
ax.set_xlabel('frames (224 × 224, tubelets 2 × 16 × 16)', fontsize=15)
ax.set_ylabel('attention pairs per layer', fontsize=15)
ax.tick_params(axis='y', labelsize=13)
ax.minorticks_off()
ax.legend(fontsize=15, frameon=False)
ax.grid(alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
savefig('video_attention_cost')
plt.show()

### Factorised attention is a reshape

Attention takes a tensor of shape (batch, tokens, channels) and never mixes different batch entries. To attend only
within a frame, we move time into the batch; to attend only across time, we move the spatial positions into the batch.

In [ ]:
B, T_, H_, W_, D = 2, 8, 14, 14, 64                  # 16 frames -> 8 tubelet steps of 14 x 14 tokens
x = torch.randn(B, T_, H_, W_, D)
spatial_attention = torch.nn.MultiheadAttention(D, num_heads=4, batch_first=True)
temporal_attention = torch.nn.MultiheadAttention(D, num_heads=4, batch_first=True)

# spatial: every frame is its own batch entry
xs = x.reshape(B * T_, H_ * W_, D)
xs = spatial_attention(xs, xs, xs)[0]
x = xs.reshape(B, T_, H_, W_, D)
print('spatial: ', tuple(xs.shape), '= (B·T, H·W, D)')

# temporal: every spatial position is its own batch entry
xt = x.permute(0, 2, 3, 1, 4).reshape(B * H_ * W_, T_, D)
xt = temporal_attention(xt, xt, xt)[0]
x = xt.reshape(B, H_, W_, T_, D).permute(0, 3, 1, 2, 4)
print('temporal:', tuple(xt.shape), '= (B·H·W, T, D)')
print('output:  ', tuple(x.shape))

# which tokens one query token (red) attends to (blue), on a toy grid of 4 frames of 4 x 4 tokens
fig, axs = plt.subplots(1, 2, figsize=(18, 3.6), gridspec_kw=dict(wspace=0.08))
query = (1, 1, 2)                                     # frame 1, row 1, column 2
rules = [('spatial attention: the same frame\n(B·T) × (H·W) × D', lambda t, i, j: t == query[0]),
         ('temporal attention: the same position in every frame\n(B·H·W) × T × D', lambda t, i, j: (i, j) == query[1:])]
for ax, (title, rule) in zip(axs, rules):
    for t in range(4):
        for i in range(4):
            for j in range(4):
                is_query = (t, i, j) == query
                colour = '#d62728' if is_query else '#2a6fdb' if rule(t, i, j) else '#e4e4e4'
                ax.add_patch(plt.Rectangle((t * 5 + j, 3 - i), 0.9, 0.9, color=colour))
        ax.text(t * 5 + 1.95, -0.6, f'frame {t + 1}', ha='center', fontsize=14)
    ax.set_xlim(-0.3, 19.2); ax.set_ylim(-1.1, 4.1)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=16)
savefig('video_factorised_attention')
plt.show()

### Masked video modelling: tube masking

VideoMAE hides 90% of the tubelets and trains a transformer to reconstruct them. It masks *tubes*: the same positions in
every frame, so the model cannot copy a hidden patch from the neighbouring frame (which, as we saw, is nearly identical).

In [ ]:
# tube masking on the cat clip: 16 frames resized to 224 x 384 (14 x 24 patches of 16 x 16), 90% of the positions hidden
frames = np.stack([cv2.resize(cat[i], (224, 384), interpolation=cv2.INTER_AREA) for i in cat16])
rng = np.random.default_rng(1)
n_patches = 14 * 24
visible = np.zeros(n_patches, bool)
visible[rng.choice(n_patches, round(0.1 * n_patches), replace=False)] = True     # 10% visible
mask = np.kron(visible.reshape(24, 14), np.ones((16, 16)))[..., None].astype(bool)
masked = np.where(mask, frames, 200).astype(np.uint8)
print(f'{visible.sum()} of {n_patches} positions visible in all 16 frames: {visible.mean():.0%}')

show = [0, 7, 15]
fig, axs = plt.subplots(1, 7, figsize=(21, 6.2), gridspec_kw=dict(width_ratios=[1, 1, 1, 0.15, 1, 1, 1], wspace=0.06))
for ax, i in zip(axs[:3], show):
    ax.imshow(frames[i]); ax.set_title(f'frame {i + 1} of 16', fontsize=17)
for ax, i in zip(axs[4:], show):
    ax.imshow(masked[i]); ax.set_title(f'frame {i + 1}, masked', fontsize=17)
for ax in axs:
    ax.axis('off')
savefig('video_tube_masking')
plt.show()

## Video and text

X-CLIP (Ni et al., ECCV 2022) is a video-text model: a CLIP image encoder whose frame features exchange information
across time (cross-frame attention, a small temporal transformer) before they are compared with the text. The checkpoint
we use was trained on Kinetics-400 with 8 frames. For each clip we compute the probability of each caption (a softmax
over the video-text similarities): retrieval works for our four clips.

In [ ]:
import textwrap
from transformers import XCLIPModel, XCLIPProcessor
xclip = XCLIPModel.from_pretrained('microsoft/xclip-base-patch32').to(device).eval()
xclip_processor = XCLIPProcessor.from_pretrained('microsoft/xclip-base-patch32')

def caption_probs(frames, captions, n=8):
    # probability of each caption for a clip (n frames sampled uniformly, kept in the given order)
    picked = frames[np.linspace(0, len(frames) - 1, n).round().astype(int)]
    inputs = xclip_processor(text=captions, videos=[list(picked)], return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        return xclip(**inputs).logits_per_video.softmax(dim=-1)[0].cpu().numpy()

poodles, poodles_fps = read_video('poodles.mp4')
clips = {'running': running, 'cat': cat, 'synthetic dog': video, 'poodles': poodles}
captions = ['a man jogging along a street', 'a cat opening a door', 'a dog walking past potted plants',
            'two dogs playing on a lawn']
probs_matrix = np.stack([caption_probs(frames, captions) for frames in clips.values()])
print(np.round(probs_matrix, 3))

fig = plt.figure(figsize=(15, 6.2))
gs = fig.add_gridspec(len(clips), 2, width_ratios=[0.5, 2.6], wspace=0.04, hspace=0.08)
for k, frames in enumerate(clips.values()):
    ax = fig.add_subplot(gs[k, 0])
    ax.imshow(frames[len(frames) // 2]); ax.axis('off')
ax = fig.add_subplot(gs[:, 1])
ax.imshow(probs_matrix, cmap='Blues', aspect='auto', vmin=0, vmax=1)
for i in range(len(clips)):
    for j in range(len(captions)):
        best = probs_matrix[i].argmax() == j
        ax.text(j, i, f'{probs_matrix[i, j]:.2f}', ha='center', va='center', fontsize=17,
                color='white' if probs_matrix[i, j] > 0.5 else 'black', fontweight='bold' if best else 'normal')
ax.set_xticks(range(len(captions)))
ax.set_xticklabels([textwrap.fill(c, 18) for c in captions], fontsize=15)
ax.xaxis.tick_top()
ax.set_yticks([])
savefig('video_text_similarity')
plt.show()

### Does the video-text model use time?

The same test as for VideoMAE: the cat clip forwards and backwards, against captions that differ only in the direction.

In [ ]:
time_captions = ['a cat opening a door', 'a cat closing a door', 'a dog playing on a lawn', 'a man jogging along a street']
p_forward = caption_probs(cat, time_captions)
p_reversed = caption_probs(cat[::-1], time_captions)
for c, f_, r_ in zip(time_captions, p_forward, p_reversed):
    print(f'{c:30s} forward {f_:.3f}   reversed {r_:.3f}')

pick = np.linspace(0, len(cat) - 1, 8).round().astype(int)
fig = plt.figure(figsize=(20, 6.4))
gs = fig.add_gridspec(2, 9, width_ratios=[1] * 8 + [3.2], wspace=0.05, hspace=0.25)
for row, (name, order, p) in enumerate([('forward', pick, p_forward), ('reversed', pick[::-1], p_reversed)]):
    for k, i in enumerate(order):
        ax = fig.add_subplot(gs[row, k])
        ax.imshow(cat[i][96:]); ax.axis('off')
        if k == 0:
            ax.set_title(name, fontsize=18, loc='left')
    ax = fig.add_subplot(gs[row, 8])
    ax.barh([1, 0], p[:2], color=['#2a6fdb', '#e07b28'], height=0.6)
    for y, v, c in zip([1, 0], p[:2], time_captions[:2]):
        ax.text(0.02, y, c.split()[2], va='center', fontsize=18, color='white')   # 'opening' / 'closing'
        ax.text(v + 0.02, y, f'{v:.2f}', va='center', fontsize=18)
    ax.set_xlim(0, 1); ax.set_yticks([]); ax.tick_params(axis='x', labelsize=12)
    ax.spines[['top', 'right', 'left']].set_visible(False)
savefig('video_text_time')
plt.show()

## Video in language models: token budgets

A video language model encodes sampled frames with an image encoder and feeds the visual tokens to a language model.
With a typical 256 tokens per frame the context fills up quickly, so frames are sampled sparsely and tokens are pooled.

In [ ]:
# sparse sampling: 8 frames of the cat clip (3.8 s), 256 tokens each
sampled = np.linspace(0, len(cat) - 1, 8).round().astype(int)
fig, axs = plt.subplots(1, 8, figsize=(20, 5.4), gridspec_kw=dict(wspace=0.05))
for ax, i in zip(axs, sampled):
    ax.imshow(cat[i]); ax.axis('off')
    ax.set_title(f'{i / cat_fps:.1f} s', fontsize=16)
    ax.text(0.5, -0.06, '256 tokens', transform=ax.transAxes, ha='center', va='top', fontsize=15, color='#2a6fdb')
savefig('video_vlm_sampling')
plt.show()

def video_tokens(seconds, fps=1.0, tokens_per_frame=256):
    return seconds * fps * tokens_per_frame

for name, seconds in [('1 min', 60), ('10 min', 600), ('1 h', 3600)]:
    print(f'{name:6s}: {video_tokens(seconds):>10,.0f} tokens at 1 FPS x 256, '
          f'{video_tokens(seconds, tokens_per_frame=64):>9,.0f} pooled to 64 per frame')

# tokens against duration, compared with context lengths of language models
seconds = np.logspace(1, np.log10(4 * 3600), 100)
fig, ax = plt.subplots(figsize=(13, 4.8))
for fps, tpf, colour in [(1, 256, '#2a6fdb'), (1, 64, '#e07b28'), (0.1, 64, '#2ca02c')]:
    ax.loglog(seconds / 60, video_tokens(seconds, fps, tpf), color=colour, lw=2.5, label=f'{fps:g} FPS × {tpf} tokens')
for context in [32_000, 128_000, 1_000_000]:
    ax.axhline(context, color='grey', ls='--', lw=1)
    ax.text(0.18, context * 1.12, f'context {context // 1000}k tokens' if context < 1e6 else 'context 1M tokens', fontsize=13, color='grey')
ax.set_xlabel('video length', fontsize=15)
ax.set_ylabel('visual tokens', fontsize=15)
ax.set_xticks([1 / 6, 1, 10, 60, 240])
ax.set_xticklabels(['10 s', '1 min', '10 min', '1 h', '4 h'], fontsize=13)
ax.tick_params(axis='y', labelsize=13)
ax.minorticks_off()
ax.legend(fontsize=14, frameon=False, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
savefig('video_vlm_tokens')
plt.show()

## Asking a video VLM

Qwen2.5-VL-3B-Instruct (Bai et al., 2025) is a small video language model. We give it the 8 frames of the cat clip
(forwards, then backwards) and ask two questions.

**This needs about 8 GB of GPU memory and a 7.5 GB download (a Colab T4 is enough, but it takes a few minutes).**
By default (`RUN_VLM = False`) the next cell loads our answers from `vlm_cat_answers.json`; set it to `True` to run
the model yourself.

In [ ]:
import json
RUN_VLM = False   # True: download and run Qwen2.5-VL-3B (about 8 GB of GPU memory)
QUESTIONS = ['Is the cat opening or closing the door? Answer in one short sentence.',
             'What happens in this video? Answer in one sentence.']
vlm_frames = np.linspace(0, len(cat) - 1, 8).round().astype(int)

if RUN_VLM:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    name = 'Qwen/Qwen2.5-VL-3B-Instruct'
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(name, dtype=dtype, device_map=device).eval()
    vlm_processor = AutoProcessor.from_pretrained(name)

    def ask(frames, question):
        messages = [{'role': 'user', 'content': [{'type': 'video'}, {'type': 'text', 'text': question}]}]
        text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = vlm_processor(text=[text], videos=[frames], return_tensors='pt',
                               videos_kwargs=dict(fps=2.0, do_sample_frames=False)).to(device)
        with torch.no_grad():
            out = vlm.generate(**inputs, max_new_tokens=60, do_sample=False)
        return vlm_processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()

    answers = {order: {q: ask(cat[idx], q) for q in QUESTIONS}
               for order, idx in [('forward', vlm_frames), ('reversed', vlm_frames[::-1])]}
    json.dump(answers, open('vlm_cat_answers.json', 'w'), indent=1)
else:
    print('RUN_VLM = False: loading the saved answers from vlm_cat_answers.json')
    answers = json.load(open('vlm_cat_answers.json'))

for order in answers:
    for q, a in answers[order].items():
        print(f'{order:9s} Q: {q}\n          A: {a}')

fig = plt.figure(figsize=(15.5, 7.6))
gs = fig.add_gridspec(2, 8, wspace=0.04, hspace=0.62)
for row, (order, idx) in enumerate([('forward', vlm_frames), ('reversed', vlm_frames[::-1])]):
    for k, i in enumerate(idx):
        ax = fig.add_subplot(gs[row, k])
        ax.imshow(cat[i][96:]); ax.axis('off')
        if k == 0:
            ax.set_title(order, fontsize=22, loc='left')
            first = ax
    a1, a2 = answers[order].values()
    first.text(0, -0.05, f'“Opening or closing?”  →  {a1}\n“What happens?”  →  {a2}', transform=first.transAxes,
               fontsize=22, va='top', ha='left')
savefig('video_vlm_answers')
plt.show()